In [1]:
%pip install scikit-learn -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install ucimlrepo -q

In [3]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

spam = fetch_ucirepo(id=94)    # 94 = Spambase on UCI
df = pd.concat([spam.data.features, spam.data.targets], axis=1)
df = df.rename(columns={df.columns[-1]: "is_spam"})
print(df.shape)
df["is_spam"].value_counts()

(4601, 58)


is_spam
0    2788
1    1813
Name: count, dtype: int64

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="is_spam")   # the 57 features
y = df["is_spam"]                # the answer
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (3680, 57)  Test: (921, 57)


In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes": GaussianNB(),
    "SVM": make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
}

In [6]:
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)                 # learn from training data
    pred = model.predict(X_test)                # 0/1 guesses on test data
    prob = model.predict_proba(X_test)[:, 1]    # probability of spam
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, prob),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "MCC": matthews_corrcoef(y_test, pred),
    })

results = pd.DataFrame(rows).round(4)
results

/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9294,0.9702,0.9209,0.8981,0.9093,0.8518
1,Decision Tree,0.9034,0.9154,0.9029,0.8457,0.8734,0.7965
2,KNN,0.9077,0.9506,0.8861,0.8788,0.8824,0.8065
3,Naive Bayes,0.8339,0.9449,0.7178,0.9532,0.8189,0.6941
4,SVM,0.9273,0.9667,0.9277,0.8843,0.9055,0.8471


In [13]:
import os, sys, joblib, sklearn, numpy as np

os.makedirs("project/model", exist_ok=True)
for name, model in models.items():
    joblib.dump(model, "project/model/" + name.lower().replace(" ", "_") + ".pkl")

test_df = X_test.copy()
test_df["is_spam"] = y_test
test_df.to_csv("project/test_data.csv", index=False)

with open("project/requirements.txt", "w") as f:
    f.write("streamlit\n")
    f.write("scikit-learn==" + sklearn.__version__ + "\n")
    f.write("numpy==" + np.__version__ + "\n")
    f.write("pandas\njoblib\nmatplotlib\nseaborn\n")

print("Saved:", os.listdir("project"), os.listdir("project/model"))
print("PYTHON VERSION:", sys.version.split()[0])

Saved: ['model', 'test_data.csv', 'requirements.txt'] ['logistic_regression.pkl', 'decision_tree.pkl', 'knn.pkl', 'naive_bayes.pkl', 'svm.pkl']
PYTHON VERSION: 3.12.7


In [15]:
print("| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |")
print("|---|---|---|---|---|---|---|")
for _, r in results.iterrows():
    print(f"| {r['Model']} | {r['Accuracy']} | {r['AUC']} | {r['Precision']} | {r['Recall']} | {r['F1']} | {r['MCC']} |")

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
|---|---|---|---|---|---|---|
| Logistic Regression | 0.9294 | 0.9702 | 0.9209 | 0.8981 | 0.9093 | 0.8518 |
| Decision Tree | 0.9034 | 0.9154 | 0.9029 | 0.8457 | 0.8734 | 0.7965 |
| KNN | 0.9077 | 0.9506 | 0.8861 | 0.8788 | 0.8824 | 0.8065 |
| Naive Bayes | 0.8339 | 0.9449 | 0.7178 | 0.9532 | 0.8189 | 0.6941 |
| SVM | 0.9273 | 0.9667 | 0.9277 | 0.8843 | 0.9055 | 0.8471 |
